# Hard-to-distinguish biologically related cell-type subtypes

This notebook is the extracted hard-cell-type section from `02_disentangle.ipynb`. It reads canonical full-data benchmark, disentanglement, and single-cell recovery outputs and does not fit or alter any model.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
suppressPackageStartupMessages({
  library(here); library(dplyr); library(tidyr); library(ggplot2); library(patchwork)
})

## Settings

In [ ]:
OUT_DIR <- here(snakemake@params[["out_dir"]])
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

top_lvs_df <- read.csv(snakemake@input[["top"]], stringsAsFactors = FALSE)
lv_corr_full <- read.csv(snakemake@input[["corr"]], stringsAsFactors = FALSE)
selected_all_df <- read.csv(snakemake@input[["selected"]], stringsAsFactors = FALSE)
top_pathway_df <- read.csv(snakemake@input[["pathway_recovery"]], stringsAsFactors = FALSE)
enr_all <- read.csv(snakemake@input[["enrichment"]], stringsAsFactors = FALSE)
lv_effect_all <- read.csv(snakemake@input[["effects"]], stringsAsFactors = FALSE)
recovery_df <- read.csv(snakemake@input[["recovery"]], stringsAsFactors = FALSE)

module_results <- lapply(split(selected_all_df, selected_all_df$dataset),
                         function(x) list(selected = x))
Z_LIM <- max(abs(lv_effect_all$row_effect), na.rm = TRUE)
recovered_top_q <- -log10(top_pathway_df$top1_fdr[top_pathway_df$recovered %in% TRUE] + 1e-300)
Q_LIM <- max(c(enr_all$neg_log10_fdr, recovered_top_q), na.rm = TRUE)

ct_labels_df <- read.csv(here('data', 'pseudobulk', 'cell_type_labels.csv'),
                         stringsAsFactors = FALSE)
CT_LABELS <- setNames(ct_labels_df$label, ct_labels_df$cell_type)
ct_label <- function(x) ifelse(x %in% names(CT_LABELS), CT_LABELS[x], x)

## Hard to distinguish biologically related cell-type subtypes

In [ ]:
RELATED_GROUPS <- list(
    list(entries = data.frame(ds = c("PBMC_Perez2022", "PBMC_Perez2022"),
                              ct = c("B_cell", "T_cell"), stringsAsFactors = FALSE)),
    list(entries = data.frame(ds = c("PBMC_1k1k", "PBMC_1k1k"),
                              ct = c("CD4_T", "CD8_T"), stringsAsFactors = FALSE)),
    list(entries = data.frame(ds = c("Brain_Xiong2023", "Brain_Xiong2023"),
                              ct = c("Opc", "Oli"), stringsAsFactors = FALSE)),
    list(entries = data.frame(ds = c("Brain_Mathys2023", "Brain_Mathys2023"),
                              ct = c("Ast", "Mic"), stringsAsFactors = FALSE)),
    list(entries = data.frame(ds = c("Brain_Mathys2023", "Brain_Xiong2023"),
                              ct = c("Inh", "Exc"), stringsAsFactors = FALSE)),
    list(entries = data.frame(ds = c("Heart_Datar2026", "Heart_Datar2026"),
                              ct = c("AtrialCM", "VentricularCM"), stringsAsFactors = FALSE))
)

wrap_panel_title <- function(x) paste(strwrap(x, width = 25), collapse = "\n")

make_group_heatmap_panel <- function(grp) {
    entries <- grp$entries
    entries$entry_id <- paste(entries$ds, entries$ct, sep = "__")

    assigned <- dplyr::bind_rows(lapply(seq_len(nrow(entries)), function(i) {
        row <- top_lvs_df[top_lvs_df$dataset == entries$ds[i] &
                          top_lvs_df$cell_type == entries$ct[i], ]
        if (nrow(row) == 0) return(NULL)
        data.frame(entry_id = entries$entry_id[i], ds = entries$ds[i], ct = entries$ct[i],
                  LV = row$LV[1], stringsAsFactors = FALSE)
    }))
    if (nrow(assigned) < 2) return(NULL)

    row_labels <- setNames(ct_label(assigned$ct), assigned$entry_id)
    col_labels <- setNames(assigned$LV, assigned$entry_id)

    sub <- dplyr::bind_rows(lapply(seq_len(nrow(assigned)), function(i) {
        same_ds  <- assigned[assigned$ds == assigned$ds[i], ]
        cor_rows <- lv_corr_full[lv_corr_full$dataset == assigned$ds[i] &
                                 lv_corr_full$LV == assigned$LV[i] &
                                 lv_corr_full$cell_type %in% same_ds$ct, ]
        if (nrow(cor_rows) == 0) return(NULL)
        cor_rows$col_entry_id <- assigned$entry_id[i]
        cor_rows$row_entry_id <- same_ds$entry_id[match(cor_rows$cell_type, same_ds$ct)]
        cor_rows
    }))
    if (is.null(sub) || nrow(sub) == 0) return(NULL)

    sub$cell_type_label <- factor(row_labels[sub$row_entry_id], levels = unname(row_labels))
    sub$LV_label        <- factor(col_labels[sub$col_entry_id], levels = unname(col_labels))

    ggplot(sub, aes(x = cell_type_label, y = LV_label, fill = cor)) +
        geom_tile(color = "white", linewidth = 0.5) +
        geom_text(aes(label = sprintf("%.2f", cor)), size = 3.2) +
        scale_fill_gradientn(colours = c("#b2182b", "#fdf7f7", "white", "#f7fbf7", "#007a33"),
                             values = scales::rescale(c(-1, -0.5, 0, 0.5, 1)),
                             limits = c(-1, 1), name = "r") +
        coord_fixed() +
        theme_bw(base_size = 10) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1),
              plot.title  = element_text(face = "bold", size = 10, hjust = 0.5)) +
        labs(x = NULL, y = NULL,
             title = wrap_panel_title(paste(unname(row_labels), collapse = " vs. ")))
}

group_panels <- Filter(Negate(is.null), lapply(RELATED_GROUPS, make_group_heatmap_panel))

options(repr.plot.width = 12, repr.plot.height = 16)
p_groups <- wrap_plots(group_panels, ncol = 2, guides = "collect") &
    theme(legend.position = "right")
print(p_groups)

In [ ]:
TOP_PATHWAY_LABEL_OVERRIDES <- c("Heart_Datar2026__AtrialCM" = "Atrial cell")

make_group_dot_panel <- function(grp, group_id) {
    entries <- grp$entries
    entries$entry_id <- paste(entries$ds, entries$ct, sep = "__")

    sel <- dplyr::bind_rows(lapply(seq_len(nrow(entries)), function(i) {
        ds  <- entries$ds[i]
        ct  <- entries$ct[i]
        res <- module_results[[ds]]
        if (is.null(res)) return(NULL)
        row <- res$selected[res$selected$cell_type == ct, ]
        if (nrow(row) == 0) return(NULL)
        row <- row[1, , drop = FALSE]
        row$entry_dataset   <- ds
        row$entry_cell_type <- ct
        row$entry_id        <- entries$entry_id[i]
        row
    }))
    if (nrow(sel) < 2) return(NULL)

    row_labels <- setNames(ct_label(entries$ct), entries$entry_id)
    tp_for_sel <- dplyr::bind_rows(lapply(seq_len(nrow(sel)), function(i) {
        tp <- top_pathway_df[top_pathway_df$dataset == sel$entry_dataset[i] &
                             top_pathway_df$cell_type == sel$entry_cell_type[i], ]
        if (nrow(tp) == 0) return(data.frame(
            entry_id = sel$entry_id[i], top_pathway_label = NA_character_,
            recovered = FALSE, top1_fdr = NA_real_, stringsAsFactors = FALSE))
        data.frame(entry_id = sel$entry_id[i], top_pathway_label = tp$top_pathway_label[1],
                   recovered = tp$recovered[1], top1_fdr = tp$top1_fdr[1],
                   stringsAsFactors = FALSE)
    }))
    tp_label <- setNames(tp_for_sel$top_pathway_label, tp_for_sel$entry_id)
    override_idx <- names(tp_label) %in% names(TOP_PATHWAY_LABEL_OVERRIDES)
    tp_label[override_idx] <- TOP_PATHWAY_LABEL_OVERRIDES[names(tp_label)[override_idx]]
    tp_label <- sub(" - .*$", "", tp_label)
    col_labels <- setNames(
        ifelse(is.na(tp_label[sel$entry_id]) | tp_label[sel$entry_id] == sel$LV,
               sel$LV, paste0(tp_label[sel$entry_id], " - ", sel$LV)),
        sel$entry_id)
    recovered_by_entry <- setNames(tp_for_sel$recovered, tp_for_sel$entry_id)
    top1_neg_log10_by_entry <- setNames(-log10(tp_for_sel$top1_fdr + 1e-300),
                                        tp_for_sel$entry_id)

    enr <- dplyr::bind_rows(lapply(seq_len(nrow(sel)), function(i) {
        marker_cts <- entries$ct[entries$ds == sel$entry_dataset[i]]
        sub <- enr_all[enr_all$dataset == sel$entry_dataset[i] &
                       enr_all$LV == sel$LV[i] &
                       enr_all$lv_cell_type == sel$entry_cell_type[i] &
                       enr_all$marker_cell_type %in% marker_cts, ]
        if (nrow(sub) == 0) return(NULL)
        sub$lv_entry_id     <- sel$entry_id[i]
        sub$marker_entry_id <- paste(sub$dataset, sub$marker_cell_type, sep = "__")
        sub
    }))
    if (!all(c("marker_entry_id", "lv_entry_id") %in% names(enr))) {
        enr <- data.frame(marker_entry_id = character(), lv_entry_id = character(),
                          neg_log10_fdr = numeric(), row_effect = numeric(),
                          stringsAsFactors = FALSE)
    }

    grid <- expand.grid(marker_entry_id = entries$entry_id,
                        lv_entry_id = sel$entry_id,
                        stringsAsFactors = FALSE)
    enr <- merge(grid, enr, by = c("marker_entry_id", "lv_entry_id"), all.x = TRUE)
    enr$neg_log10_fdr[is.na(enr$neg_log10_fdr)] <- 0

    effect_lookup <- dplyr::bind_rows(lapply(seq_len(nrow(sel)), function(i) {
        marker_entries <- entries[entries$ds == sel$entry_dataset[i], ]
        eff <- lv_effect_all[lv_effect_all$dataset == sel$entry_dataset[i] &
                             lv_effect_all$LV == sel$LV[i] &
                             lv_effect_all$marker_cell_type %in% marker_entries$ct, ]
        if (nrow(eff) == 0) return(NULL)
        eff$lv_entry_id     <- sel$entry_id[i]
        eff$marker_entry_id <- paste(eff$dataset, eff$marker_cell_type, sep = "__")
        eff[, c("marker_entry_id", "lv_entry_id", "row_effect")]
    }))
    enr$row_effect <- NULL
    if (nrow(effect_lookup) > 0)
        enr <- merge(enr, effect_lookup, by = c("marker_entry_id", "lv_entry_id"), all.x = TRUE)
    if (!"row_effect" %in% names(enr)) enr$row_effect <- NA_real_
    enr$row_effect[is.na(enr$row_effect)] <- 0

    recover_idx <- enr$marker_entry_id == enr$lv_entry_id &
                   recovered_by_entry[enr$lv_entry_id] %in% TRUE &
                   !is.na(top1_neg_log10_by_entry[enr$lv_entry_id])
    enr$neg_log10_fdr[recover_idx] <- pmax(enr$neg_log10_fdr[recover_idx],
                                           top1_neg_log10_by_entry[enr$lv_entry_id[recover_idx]])
    enr$marker_row_label <- factor(row_labels[enr$marker_entry_id], levels = unname(row_labels))
    enr$lv_col_label     <- factor(col_labels[enr$lv_entry_id], levels = unname(col_labels))

    panel_title <- wrap_panel_title(paste(unname(row_labels), collapse = " vs. "))

    panel_plot <- ggplot(enr, aes(x = lv_col_label, y = marker_row_label)) +
        geom_point(aes(size = neg_log10_fdr, color = row_effect)) +
        scale_size_continuous(limits = c(0, Q_LIM), range = c(1, 9),
                              name = expression("-" * log[10] ~ "(FDR)")) +
        scale_color_gradientn(colors = c("#b2182b", "#f4a582", "#f7f7f7", "#a1d99b", "#007a33"),
                              values = c(0, 0.35, 0.5, 0.65, 1),
                              limits = c(-Z_LIM, Z_LIM),
                              name = "LV effect") +
        coord_fixed() +
        theme_bw(base_size = 9) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1),
              plot.title  = element_text(face = "bold", size = 10, hjust = 0.5)) +
        labs(x = NULL, y = NULL, title = panel_title)

    panel_export <- data.frame(
        group_id         = group_id,
        source_datasets  = paste(unique(entries$ds), collapse = ";"),
        panel_title      = panel_title,
        marker_row_label = as.character(enr$marker_row_label),
        marker_row_rank  = as.integer(enr$marker_row_label),
        lv_col_label     = as.character(enr$lv_col_label),
        lv_col_rank      = as.integer(enr$lv_col_label),
        neg_log10_fdr    = enr$neg_log10_fdr,
        row_effect       = enr$row_effect,
        Z_LIM            = Z_LIM,
        Q_LIM            = Q_LIM,
        stringsAsFactors = FALSE
    )

    list(plot = panel_plot, data = panel_export)
}

group_dot_panels <- Filter(Negate(is.null), lapply(
    seq_along(RELATED_GROUPS),
    function(i) make_group_dot_panel(RELATED_GROUPS[[i]], i)
))

group_dot_export <- dplyr::bind_rows(lapply(group_dot_panels, `[[`, "data"))

options(repr.plot.width = 12, repr.plot.height = 15)
p_group_dots <- wrap_plots(lapply(group_dot_panels, `[[`, "plot"), ncol = 2, guides = "collect") &
    theme(legend.position = "right")
print(p_group_dots)

## Export canonical hard-cell results

In [ ]:
hard_entries <- dplyr::bind_rows(lapply(seq_along(RELATED_GROUPS), function(i) {
  entries <- RELATED_GROUPS[[i]]$entries
  entries$group_id <- i
  entries
}))

hard_table <- hard_entries %>%
  left_join(top_lvs_df, by = c('ds' = 'dataset', 'ct' = 'cell_type')) %>%
  left_join(recovery_df, by = c('ds' = 'dataset', 'ct' = 'cell_type'),
            suffix = c('_specificity', '_singlecell'))

hard_corr_long <- dplyr::bind_rows(lapply(seq_len(nrow(hard_entries)), function(i) {
  entry <- hard_entries[i, ]
  assigned <- top_lvs_df[top_lvs_df$dataset == entry$ds & top_lvs_df$cell_type == entry$ct, ]
  if (!nrow(assigned)) return(NULL)
  candidates <- hard_entries$ct[hard_entries$ds == entry$ds]
  out <- lv_corr_full[lv_corr_full$dataset == entry$ds &
                      lv_corr_full$LV == assigned$LV[1] &
                      lv_corr_full$cell_type %in% candidates, ]
  if (!nrow(out)) return(NULL)
  out$assigned_cell_type <- entry$ct
  out$group_id <- entry$group_id
  out
}))

hard_enrichment_long <- dplyr::bind_rows(lapply(seq_len(nrow(hard_entries)), function(i) {
  entry <- hard_entries[i, ]
  assigned <- top_lvs_df[top_lvs_df$dataset == entry$ds & top_lvs_df$cell_type == entry$ct, ]
  if (!nrow(assigned)) return(NULL)
  candidates <- hard_entries$ct[hard_entries$ds == entry$ds]
  out <- enr_all[enr_all$dataset == entry$ds & enr_all$LV %in% assigned$LV &
                 enr_all$marker_cell_type %in% candidates, ]
  if (!nrow(out)) return(NULL)
  out$group_id <- entry$group_id
  out
}))

write.csv(hard_table, file.path(OUT_DIR, 'hard_cell_types.csv'), row.names = FALSE)
write.csv(hard_corr_long, file.path(OUT_DIR, 'related_lv_correlations.csv'), row.names = FALSE)
write.csv(hard_enrichment_long, file.path(OUT_DIR, 'hard_marker_enrichment.csv'), row.names = FALSE)
write.csv(group_dot_export, file.path(OUT_DIR, 'hard_group_dot_ready.csv'), row.names = FALSE)
cat("Saved hard-cell-type outputs to", OUT_DIR, "\n")